# Google Ajou AI Capstone Base Notebook

팀원이 Colab 또는 로컬에서 개인 폴더의 코드를 쉽게 실행하기 위한 최소 스켈레톤입니다.

사용자는 아래 두 값만 바꾸면 됩니다.

```python
USER_FOLDER = "SangHyo"
RUN_FILE = "src/train.py"
```

- `USER_FOLDER`: repo 안의 개인 폴더명
- `RUN_FILE`: 개인 폴더 기준으로 실행할 Python 파일
- 개인 폴더에 `requirements.txt` 또는 `requirement.txt`가 있으면 자동 설치합니다.
- 실행 파일에 `main(config)`가 있으면 `main(config)`를 호출합니다.
- `main(config)`가 없으면 일반 Python script처럼 실행합니다.


## 셀 1. 기본 환경 준비

Colab이면 Google Drive를 마운트하고, repo 위치와 공통 데이터 위치를 잡습니다. repo가 없는 Colab 환경에서는 GitHub에서 clone합니다.


In [ ]:
# Cell 1 - Environment setup
import os
import runpy
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Pig30nidaE/Google-Ajou-AICapstone.git"
REPO_DIR_NAME = "Google-Ajou-AICapstone"


def in_colab():
    try:
        import google.colab  # type: ignore
        return True
    except Exception:
        return False


IN_COLAB = in_colab()

if IN_COLAB:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive")


def find_project_root():
    cwd = Path.cwd().resolve()
    for path in [cwd, *cwd.parents]:
        if (path / "base.ipynb").exists() or (path / "Data").exists():
            return path
    return None


PROJECT_ROOT = find_project_root()

if PROJECT_ROOT is None and IN_COLAB:
    clone_path = Path("/content") / REPO_DIR_NAME
    if not clone_path.exists():
        subprocess.run(["git", "clone", REPO_URL, str(clone_path)], check=True)
    PROJECT_ROOT = clone_path

if PROJECT_ROOT is None:
    PROJECT_ROOT = Path.cwd().resolve()

os.chdir(PROJECT_ROOT)

if IN_COLAB:
    DATA_ROOT = Path("/content/drive/Shareddrives/GoogleAI_contest/Data")
else:
    DATA_ROOT = PROJECT_ROOT / "Data"

if not DATA_ROOT.exists() and (PROJECT_ROOT / "Data").exists():
    DATA_ROOT = PROJECT_ROOT / "Data"

print(f"IN_COLAB     : {IN_COLAB}")
print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"DATA_ROOT    : {DATA_ROOT}")


## 셀 2. 사용자 입력

여기만 수정하면 됩니다. `RUN_FILE`은 `USER_FOLDER` 기준 상대경로로 쓰는 것을 권장합니다.


In [ ]:
# Cell 2 - User inputs
USER_FOLDER = "SangHyo"
RUN_FILE = "src/train.py"


## 셀 3. 경로 확인

개인 폴더와 실행할 파일 경로를 확인합니다. 파일이 없으면 여기에서 바로 에러가 납니다.


In [ ]:
# Cell 3 - Resolve paths
USER_ROOT = (PROJECT_ROOT / USER_FOLDER).resolve()
RUN_PATH = Path(RUN_FILE).expanduser()

if not RUN_PATH.is_absolute():
    RUN_PATH = USER_ROOT / RUN_PATH

RUN_PATH = RUN_PATH.resolve()

if not USER_ROOT.exists():
    raise FileNotFoundError(f"User folder does not exist: {USER_ROOT}")

if not RUN_PATH.exists():
    raise FileNotFoundError(f"Run file does not exist: {RUN_PATH}")

print(f"USER_ROOT : {USER_ROOT}")
print(f"RUN_PATH  : {RUN_PATH}")


## 셀 4. 개인 requirements 설치

개인 폴더에 `requirements.txt` 또는 `requirement.txt`가 있으면 설치하고, 없으면 스킵합니다.


In [ ]:
# Cell 4 - Install user requirements if present
requirements_files = [
    USER_ROOT / "requirements.txt",
    USER_ROOT / "requirement.txt",
]

requirements_files = [path for path in requirements_files if path.exists()]

if requirements_files:
    for requirements_file in requirements_files:
        print(f"Installing: {requirements_file}")
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-r", str(requirements_file)],
            check=True,
        )
else:
    print("No requirements file found. Skip install.")


## 셀 5. 개인 코드 실행

`main(config)` 함수가 있으면 그 함수를 실행합니다. 없으면 일반 Python script처럼 실행합니다. 개인 코드에서는 `config["data_root"]`, `config["user_root"]` 등을 꺼내 쓰면 됩니다.


In [ ]:
# Cell 5 - Run selected file
import ast
import importlib.util

config = {
    "project_root": PROJECT_ROOT,
    "data_root": DATA_ROOT,
    "user_root": USER_ROOT,
    "run_path": RUN_PATH,
    "in_colab": IN_COLAB,
}

source = RUN_PATH.read_text(encoding="utf-8")
tree = ast.parse(source, filename=str(RUN_PATH))
has_main = any(
    isinstance(node, ast.FunctionDef) and node.name == "main"
    for node in tree.body
)

if has_main:
    spec = importlib.util.spec_from_file_location("user_run_file", RUN_PATH)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    print("Run mode: main(config)")
    result = module.main(config)
else:
    print("Run mode: script")
    result = runpy.run_path(
        str(RUN_PATH),
        init_globals={
            "config": config,
            "PROJECT_ROOT": PROJECT_ROOT,
            "DATA_ROOT": DATA_ROOT,
            "USER_ROOT": USER_ROOT,
        },
        run_name="__main__",
    )

print("Done.")


## 개인 코드 예시

`SangHyo/src/train.py` 예시:

```python
from pathlib import Path


def main(config):
    data_root = Path(config["data_root"])
    user_root = Path(config["user_root"])

    train_activity = data_root / "1.Training/SourceData/1.Gait/train_activity.csv"
    print(train_activity)
    print(user_root)
```
